# Analisi Metriche Globali
Caricamento e visualizzazione delle metriche dai file global_summary.txt

In [1]:
import pandas as pd
import os
from pathlib import Path
import re

In [2]:
# Definisci i percorsi dei file global_summary.txt
summary_files = [
    "./output/SD_Inpainting/full_dataset/VAE_MSE_UNTARGET/global_summary.txt",
    "./output/SD_Img2Img/full_dataset/VAE_MSE_UNTARGET/global_summary.txt",
    "./output/InstructionPix2Pix/full_dataset/VAE_MSE_UNTARGET/global_summary.txt",
    "./output/InstructionPix2Pix/full_dataset/VAE_MSE_FT_2_STAGE/global_summary.txt",
    "./output/SD_Inpainting/full_dataset/VAE_MSE_FT_2_STAGE/global_summary.txt",
    "./output/SD_Img2Img/full_dataset/VAE_MSE_FT_2_STAGE/global_summary.txt",
]

print(f"Cercando {len(summary_files)} file di riepilogo...")

Cercando 6 file di riepilogo...


In [3]:
def parse_global_summary(text, filename):
    """Parsa il contenuto di un file global_summary.txt"""
    data = {
        'name': filename.replace('output/', '').replace('/global_summary.txt', ''),
        'psnr_quality': None,
        'ssim_quality': None,
        'fsim_quality': None,
        'psnr_protection': None,
        'ssim_protection': None,
        'fsim_protection': None,
        'subject_lpips_orig': None,
        'global_lpips_orig': None,
        'subject_lpips_edited': None,
        'global_lpips_edited': None,
        'attack_success_score': None,
        'attack_successes_count': None,
        'attack_success_rate': None
    }

    lines = text.split('\n')
    in_orig_lpips = False
    in_edited_lpips = False

    for i, line in enumerate(lines):
        line = line.strip()

        # Parse PSNR
        if 'Average original vs immunized PSNR:' in line:
            val = line.split(':')[1].strip() if ':' in line else None
            data['psnr_quality'] = float(val) if val else None
        if 'Average edited_original vs edited_immunized PSNR:' in line:
            val = line.split(':')[1].strip() if ':' in line else None
            data['psnr_protection'] = float(val) if val else None

        # Parse SSIM
        if 'Average original vs immunized SSIM:' in line:
            val = line.split(':')[1].strip() if ':' in line else None
            data['ssim_quality'] = float(val) if val else None
        if 'Average edited_original vs edited_immunized SSIM:' in line:
            val = line.split(':')[1].strip() if ':' in line else None
            data['ssim_protection'] = float(val) if val else None

        # Parse FSIM
        if 'Average original vs immunized FSIM:' in line:
            val = line.split(':')[1].strip() if ':' in line else None
            data['fsim_quality'] = float(val) if val else None
        if 'Average edited_original vs edited_immunized FSIM:' in line:
            val = line.split(':')[1].strip() if ':' in line else None
            data['fsim_protection'] = float(val) if val else None

        # Parse LPIPS sections
        if 'Original vs Immunized LPIPS' in line:
            in_orig_lpips = True
            in_edited_lpips = False
            continue
        if 'Edited vs Adversarial LPIPS' in line:
            in_orig_lpips = False
            in_edited_lpips = True
            continue

        # Reset flags
        if line.startswith('===') or line.startswith('----'):
            if 'LPIPS' not in line:
                in_orig_lpips = False
                in_edited_lpips = False

        # Parse LPIPS Original vs Immunized
        if in_orig_lpips and line.startswith('Subject LPIPS:'):
            val = line.split(':')[1].strip() if ':' in line else None
            data['subject_lpips_orig'] = float(val) if val else None
        if in_orig_lpips and line.startswith('Global LPIPS:'):
            val = line.split(':')[1].strip() if ':' in line else None
            data['global_lpips_orig'] = float(val) if val else None

        # Parse LPIPS Edited vs Adversarial
        if in_edited_lpips and line.startswith('Subject LPIPS:'):
            val = line.split(':')[1].strip() if ':' in line else None
            data['subject_lpips_edited'] = float(val) if val else None
        if in_edited_lpips and line.startswith('Global LPIPS:'):
            val = line.split(':')[1].strip() if ':' in line else None
            data['global_lpips_edited'] = float(val) if val else None

        # Parse Qwen Attack Evaluation Summary
        if line.startswith('Average attack success score'):
            val = line.split(':')[1].strip() if ':' in line else None
            data['attack_success_score'] = float(val) if val else None
        if line.startswith('Successful attacks'):
            val = line.split(':')[1].strip() if ':' in line else None
            data['attack_successes_count'] = int(val) if val else None
        if line.startswith('Attack success rate:'):
            val = line.split(':')[1].strip() if ':' in line else None
            data['attack_success_rate'] = float(val) if val else None

    return data

print("Funzione di parsing definita")

Funzione di parsing definita


In [11]:
# Carica i dati dai file
all_data = []
loaded_files = 0

for file_path in summary_files:
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            text = f.read()
        parsed_data = parse_global_summary(text, file_path)
        all_data.append(parsed_data)
        loaded_files += 1
        print(f"✓ Caricato: {file_path}")
    else:
        print(f"✗ File non trovato: {file_path}")

print(f"\nTotale file caricati: {loaded_files}/{len(summary_files)}")

✓ Caricato: ./output/SD_Inpainting/full_dataset/VAE_MSE_UNTARGET/global_summary.txt
✓ Caricato: ./output/SD_Img2Img/full_dataset/VAE_MSE_UNTARGET/global_summary.txt
✓ Caricato: ./output/InstructionPix2Pix/full_dataset/VAE_MSE_UNTARGET/global_summary.txt
✓ Caricato: ./output/InstructionPix2Pix/full_dataset/VAE_MSE_FT_2_STAGE/global_summary.txt
✓ Caricato: ./output/SD_Inpainting/full_dataset/VAE_MSE_FT_2_STAGE/global_summary.txt
✓ Caricato: ./output/SD_Img2Img/full_dataset/VAE_MSE_FT_2_STAGE/global_summary.txt

Totale file caricati: 6/6


In [12]:
# Crea il DataFrame
df = pd.DataFrame(all_data)

# Formatta le colonne numeriche a 4 decimali
numeric_cols = df.select_dtypes(include=['float64']).columns
for col in numeric_cols:
    df[col] = df[col].apply(lambda x: f'{x:.4f}' if pd.notna(x) else '-')

print(f"\nDataFrame creato con {len(df)} righe")
df


DataFrame creato con 6 righe


,name,psnr_quality,ssim_quality,fsim_quality,psnr_protection,ssim_protection,fsim_protection,subject_lpips_orig,global_lpips_orig,subject_lpips_edited,global_lpips_edited,attack_success_score,attack_successes_count,attack_success_rate
0,./SD_Inpainting/full_dataset/VAE_MSE_UNTARGET,36.0049,0.9461,-,16.6395,0.5503,-,0.2743,0.0737,0.2743,0.4804,4.9675,134,0.6700
1,./SD_Img2Img/full_dataset/VAE_MSE_UNTARGET,36.0049,0.9461,-,22.6880,0.7821,-,0.2743,0.0737,0.5619,0.2225,4.9371,130,0.6500
2,./InstructionPix2Pix/full_dataset/VAE_MSE_UNTARGET,36.0049,0.9461,-,22.9339,0.8034,-,0.2743,0.0737,0.5060,0.1846,4.0400,94,0.4700
3,./InstructionPix2Pix/full_dataset/VAE_MSE_FT_2_STAGE,32.4316,0.9316,0.8958,22.6643,0.8052,0.6955,0.2695,0.0695,0.4702,0.1806,5.2988,137,0.6850
4,./SD_Inpainting/full_dataset/VAE_MSE_FT_2_STAGE,32.4316,0.9316,0.8958,15.7260,0.5534,0.4933,0.2695,0.0695,0.2695,0.4656,5.1721,126,0.6300
5,./SD_Img2Img/full_dataset/VAE_MSE_FT_2_STAGE,32.4316,0.9316,0.8958,inf,0.6738,-,0.2695,0.0695,0.5646,0.3745,5.8863,160,0.8000


In [13]:
# Rinomina le colonne con nomi più corti e significativi
df_renamed = df.rename(columns={
    'name': 'Esperimento',
    'psnr_quality': 'PSNR Qualità',
    'ssim_quality': 'SSIM Qualità',
    'fsim_quality': 'FSIM Qualità',
    'psnr_protection': 'PSNR Protezione',
    'ssim_protection': 'SSIM Protezione',
    'fsim_protection': 'FSIM Protezione',
    'subject_lpips_orig': 'LPIPS Sogg. Orig',
    'global_lpips_orig': 'LPIPS Glob. Orig',
    'subject_lpips_edited': 'LPIPS Sogg. Edit',
    'global_lpips_edited': 'LPIPS Glob. Edit',
    'attack_success_rate': 'Tasso Attacco'
})

print("Colonne rinominate")
df_renamed

Colonne rinominate


,Esperimento,PSNR Qualità,SSIM Qualità,FSIM Qualità,PSNR Protezione,SSIM Protezione,FSIM Protezione,LPIPS Sogg. Orig,LPIPS Glob. Orig,LPIPS Sogg. Edit,LPIPS Glob. Edit,attack_success_score,attack_successes_count,Tasso Attacco
0,./SD_Inpainting/full_dataset/VAE_MSE_UNTARGET,36.0049,0.9461,-,16.6395,0.5503,-,0.2743,0.0737,0.2743,0.4804,4.9675,134,0.6700
1,./SD_Img2Img/full_dataset/VAE_MSE_UNTARGET,36.0049,0.9461,-,22.6880,0.7821,-,0.2743,0.0737,0.5619,0.2225,4.9371,130,0.6500
2,./InstructionPix2Pix/full_dataset/VAE_MSE_UNTARGET,36.0049,0.9461,-,22.9339,0.8034,-,0.2743,0.0737,0.5060,0.1846,4.0400,94,0.4700
3,./InstructionPix2Pix/full_dataset/VAE_MSE_FT_2_STAGE,32.4316,0.9316,0.8958,22.6643,0.8052,0.6955,0.2695,0.0695,0.4702,0.1806,5.2988,137,0.6850
4,./SD_Inpainting/full_dataset/VAE_MSE_FT_2_STAGE,32.4316,0.9316,0.8958,15.7260,0.5534,0.4933,0.2695,0.0695,0.2695,0.4656,5.1721,126,0.6300
5,./SD_Img2Img/full_dataset/VAE_MSE_FT_2_STAGE,32.4316,0.9316,0.8958,inf,0.6738,-,0.2695,0.0695,0.5646,0.3745,5.8863,160,0.8000


In [14]:
# Visualizzazione con tutte le colonne
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

print("\n=== Tabella Completa (Nomi Rinominati) ===")
df


=== Tabella Completa (Nomi Rinominati) ===


,name,psnr_quality,ssim_quality,fsim_quality,psnr_protection,ssim_protection,fsim_protection,subject_lpips_orig,global_lpips_orig,subject_lpips_edited,global_lpips_edited,attack_success_score,attack_successes_count,attack_success_rate
0,./SD_Inpainting/full_dataset/VAE_MSE_UNTARGET,36.0049,0.9461,-,16.6395,0.5503,-,0.2743,0.0737,0.2743,0.4804,4.9675,134,0.6700
1,./SD_Img2Img/full_dataset/VAE_MSE_UNTARGET,36.0049,0.9461,-,22.6880,0.7821,-,0.2743,0.0737,0.5619,0.2225,4.9371,130,0.6500
2,./InstructionPix2Pix/full_dataset/VAE_MSE_UNTARGET,36.0049,0.9461,-,22.9339,0.8034,-,0.2743,0.0737,0.5060,0.1846,4.0400,94,0.4700
3,./InstructionPix2Pix/full_dataset/VAE_MSE_FT_2_STAGE,32.4316,0.9316,0.8958,22.6643,0.8052,0.6955,0.2695,0.0695,0.4702,0.1806,5.2988,137,0.6850
4,./SD_Inpainting/full_dataset/VAE_MSE_FT_2_STAGE,32.4316,0.9316,0.8958,15.7260,0.5534,0.4933,0.2695,0.0695,0.2695,0.4656,5.1721,126,0.6300
5,./SD_Img2Img/full_dataset/VAE_MSE_FT_2_STAGE,32.4316,0.9316,0.8958,inf,0.6738,-,0.2695,0.0695,0.5646,0.3745,5.8863,160,0.8000


In [15]:
# Esporta in CSV
output_file = 'metriche_globali_target_vs_untarget.csv'
df.to_csv(output_file, index=False)
print(f"✓ Esportato: {output_file}")

✓ Esportato: metriche_globali_target_vs_untarget.csv


In [8]:
# Statistiche di riepilogo
print("\n=== Statistiche ===")
print(f"Numero esperimenti: {len(df)}")
print(f"\nColonne disponibili:")
for col in df.columns:
    print(f"  - {col}")


=== Statistiche ===
Numero esperimenti: 6

Colonne disponibili:
  - name
  - psnr_quality
  - ssim_quality
  - fsim_quality
  - psnr_protection
  - ssim_protection
  - fsim_protection
  - subject_lpips_orig
  - global_lpips_orig
  - subject_lpips_edited
  - global_lpips_edited
  - attack_success_rate
